# Sparse Attention Patterns

This notebook builds a small mask library for sparse causal attention and uses graph reachability plus a toy far-token retrieval proxy to compare how quickly different patterns move information.

## Motivation

Exact causal attention gives every query access to its whole prefix, but that costs `O(T^2)` score entries for a sequence of length `T`. Sparse attention replaces the dense prefix graph with a structured subgraph that encodes an inductive bias about which dependencies matter. Local windows assume useful context is nearby; block and dilated patterns trade full coverage for cheaper routing structure; global tokens create explicit hubs; and random edges can shorten graph diameter without going fully dense. Those ideas show up in Sparse Transformers [@sparse_transformers_2019], Longformer [@longformer_2020], and BigBird [@big_bird_2020].

The research question for this notebook is narrow: given a fixed causal sparsity pattern, how quickly can information from a far source position reach the final token, and what density do we pay for that path?

## Mathematical derivation

Let `G in {0,1}^{T x T}` be a causal adjacency mask, where `G_{ij} = 1` means query position `i` may attend to key position `j`. Exact masked attention replaces the dense logits `S` with

$$\tilde{S}_{ij} = S_{ij} + \log G_{ij},$$

where `\log 0 = -\infty`, so disallowed edges vanish before softmax. The computational cost is governed by the number of nonzero mask entries, `\operatorname{nnz}(G)`, instead of `T^2`. For a local window of width `w`, `\operatorname{nnz}(G)` is about `T w`; for block attention with block size `b` and one previous block, it is about `2 T b`; for local-plus-global attention with `g` global tokens it is about `T (w + g)`; and BigBird-style local-plus-global-plus-random patterns add another `T r` term for `r` random links per query.

Stacking sparse layers turns `G` into a reachability process. Define `R^{(1)} = G` and, for `L >= 1`,

$$R^{(L+1)}_{ij} = \mathbf{1}\!\left[\sum_{k=1}^{T} R^{(L)}_{ik} G_{kj} > 0\right].$$

Then `R^{(L)}_{ij} = 1` means token `i` can depend on token `j` after `L` sparse attention layers. Because every mask below keeps the diagonal, reachability is monotone in `L`: a token can always keep what it already knows while acquiring new predecessors. The toy retrieval proxy below uses this same idea with row-normalized mask weights, so the final token's mass on the source position is a soft indicator of how easily a sparse graph carries long-range information.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.sparse_attention import (
    block_causal_mask,
    combine_masks,
    dilated_causal_mask,
    global_causal_mask,
    local_causal_mask,
    mask_density,
    random_causal_mask,
    receptive_field_sizes,
    stacked_reachability,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

## PyTorch implementation

The utility module exposes five causal masks and two analysis helpers: `stacked_reachability` computes layered dependency closure, and `receptive_field_sizes` counts how many source positions each token can access after each layer. We compare the raw patterns first, then build paper-inspired hybrids from them.

In [ ]:
length = 12
source_index = 0
target_index = length - 1
max_layers = 6

base_patterns = {
    'local': local_causal_mask(length, window_size=3),
    'block': block_causal_mask(length, block_size=3, num_previous_blocks=1),
    'dilated': dilated_causal_mask(length, dilation=2, window_size=3),
    'global': global_causal_mask(length, global_indices=[0, 6], local_window=3),
    'random': random_causal_mask(length, num_random=2, seed=7, local_window=1),
}

paper_patterns = {
    'sparse_transformer_like': combine_masks(
        block_causal_mask(length, block_size=3, num_previous_blocks=1),
        dilated_causal_mask(length, dilation=2, window_size=3),
    ),
    'longformer_like': global_causal_mask(length, global_indices=[0, 6], local_window=3),
    'bigbird_like': combine_masks(
        global_causal_mask(length, global_indices=[0, 6], local_window=3),
        random_causal_mask(length, num_random=2, seed=7, local_window=1),
    ),
}

def first_reaching_layer(mask: torch.Tensor, source: int, target: int, max_layers: int) -> int | None:
    for layer in range(1, max_layers + 1):
        if bool(stacked_reachability(mask, layer)[target, source].item()):
            return layer
    return None

def uniform_sparse_propagation(mask: torch.Tensor, values: torch.Tensor, layers: int) -> torch.Tensor:
    weights = mask.to(dtype=torch.float32)
    weights = weights / weights.sum(dim=-1, keepdim=True)
    state = values.to(dtype=torch.float32)
    for _ in range(layers):
        state = weights @ state
    return state

def far_token_summary(name: str, mask: torch.Tensor) -> dict[str, float | int | None]:
    values = torch.zeros(length, 1)
    values[source_index, 0] = 1.0
    propagated = uniform_sparse_propagation(mask, values, layers=max_layers)
    counts = receptive_field_sizes(mask, max_layers=max_layers)[:, target_index]
    return {
        'pattern': name,
        'density': round(mask_density(mask), 3),
        'fan_in_last_token': int(mask[target_index].sum().item()),
        'first_reaching_layer': first_reaching_layer(mask, source_index, target_index, max_layers),
        'target_receptive_field_sizes': [int(x) for x in counts.tolist()],
        'source_mass_after_6_layers': round(float(propagated[target_index, 0].item()), 4),
    }

base_summary = [far_token_summary(name, mask) for name, mask in base_patterns.items()]
paper_summary = [far_token_summary(name, mask) for name, mask in paper_patterns.items()]

base_summary, paper_summary

In [ ]:
dense_causal_density = float(torch.tril(torch.ones(length, length)).mean().item())

comparison = {
    'dense_causal_density': round(dense_causal_density, 3),
    'base_patterns': {row['pattern']: row for row in base_summary},
    'paper_patterns': {row['pattern']: row for row in paper_summary},
}

comparison

The delayed-copy proxy here is deliberately small: position `0` carries a unit signal, every other position starts at zero, and each sparse layer averages over its allowed predecessors. If a source is unreachable, its mass stays exactly zero. If it is reachable through many narrow paths, the mass arrives slowly and diffusely. This lets us compare graph structure without training a model.

## Numerical checks

The notebook should verify structural facts before making an interpretive claim: every mask must be causal, sparse masks should stay below the dense causal density, receptive fields should grow monotonically with depth, and the chosen toy task should expose at least one real success case and one real failure case.

In [ ]:
for mask in list(base_patterns.values()) + list(paper_patterns.values()):
    assert mask.dtype is torch.bool
    assert torch.equal(mask, torch.tril(mask))
    assert torch.all(torch.diagonal(mask))
    counts = receptive_field_sizes(mask, max_layers=max_layers)
    assert torch.all(counts[1:] >= counts[:-1])
    assert mask_density(mask) < dense_causal_density

assert comparison['base_patterns']['local']['target_receptive_field_sizes'] == [3, 5, 7, 9, 11, 12]
assert comparison['base_patterns']['block']['first_reaching_layer'] == 3
assert comparison['base_patterns']['dilated']['first_reaching_layer'] is None
assert comparison['base_patterns']['global']['first_reaching_layer'] == 1
assert comparison['paper_patterns']['bigbird_like']['first_reaching_layer'] <= comparison['paper_patterns']['longformer_like']['first_reaching_layer']
assert comparison['paper_patterns']['sparse_transformer_like']['first_reaching_layer'] == 3

comparison['base_patterns']['dilated'], comparison['paper_patterns']['bigbird_like']

## Exercises

Work the companion exercise sheet after running the notebook:

- `exercises/research/15_sparse_attention_patterns.md`
- `exercises/research/solutions/15_sparse_attention_patterns_solutions.md`

## References

- Child et al., *Generating Long Sequences with Sparse Transformers* [@sparse_transformers_2019].
- Beltagy et al., *Longformer: The Long-Document Transformer* [@longformer_2020].
- Zaheer et al., *Big Bird: Transformers for Longer Sequences* [@big_bird_2020].